# Weight-only INT8 versus palette 6-bit
Gunakan kernel SongFormer Python 3.11 di Mac. Sumbernya paket FP32 yang sudah lolos
seluruh validasi. Setiap kandidat dibuat langsung dari FP32, bukan dari kandidat lain.
Aktivasi tidak dikuantisasi; ini tidak menjanjikan komputasi integer atau speedup.
Notebook tidak mengubah encoder, anotasi, checkpoint, atau resource aplikasi.


In [ ]:
import sys, subprocess, json
from pathlib import Path
from datetime import datetime
assert Path('finetuning/config.json').is_file(), 'Gunakan root Section-Annotations'
proofs = sorted(Path('outputs.local').glob('coreml-comparison-*/fp32-validation.json'),
                key=lambda p: p.stat().st_mtime, reverse=True)
valid = []
for path in proofs:
    r = json.loads(path.read_text())
    if r.get('full_validation') and r.get('logit_parity_passed') and r.get('precision_metadata') == 'fp32':
        valid.append(path)
assert valid, 'Laporan FP32 full validation yang lolos belum ditemukan'
proof_path = valid[0]  # Bisa diganti Path(...) jika ingin memilih run tertentu.
base = json.loads(proof_path.read_text())
source = Path(base['package'])
assert source.is_dir(), f'Paket sumber tidak ada: {source}'
checkpoint = Path('outputs.local/finetuning/runs/worship-v3/best.pt')
experiment_dir = Path('outputs.local') / ('coreml-compression-' + datetime.now().strftime('%Y%m%d-%H%M%S'))
experiment_dir.mkdir(exist_ok=False)
print('Sumber FP32:', source)
print('Bukti validasi:', proof_path)
print('Output:', experiment_dir)

def run_logged(command, log_path):
    with log_path.open('w') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if code:
        raise RuntimeError(f'Proses gagal ({code}), log: {log_path}')


Buat kedua kandidat. Palette memakai k-means per grup 32 channel, satu worker untuk membatasi memori. Clustering bisa memerlukan waktu; tunggu proses selesai.

In [ ]:
for scheme in ('int8', 'palette6'):
    run_logged([sys.executable, '-u', '-m', 'deployment.compress_coreml',
        '--package', str(source), '--validation-report', str(proof_path),
        '--scheme', scheme, '--output-dir', str(experiment_dir / scheme)],
        experiment_dir / f'{scheme}-compression.log')


Periksa satu lagu dahulu. Diagnostic mode mencatat perbedaan numerik tanpa menyetujui model. Shape, NaN/Inf, hash mismatch dan runtime errors tetap menghentikan proses.

In [ ]:
for scheme in ('int8', 'palette6'):
    run_logged([sys.executable, '-u', '-m', 'deployment.verify_coreml',
        '--checkpoint', str(checkpoint),
        '--package', str(experiment_dir / scheme / 'SongFormerFunctionHead.mlpackage'),
        '--candidate-name', scheme, '--diagnostic', '--limit', '1',
        '--output', str(experiment_dir / f'{scheme}-one.json')],
        experiment_dir / f'{scheme}-one.log')


Jalankan full validation untuk kedua kandidat. Perubahan numerik pada bobot terkompresi diharapkan, tetapi dampak F1 dan kelas individual tetap harus diperiksa. Tidak memakai test.

In [ ]:
for scheme in ('int8', 'palette6'):
    run_logged([sys.executable, '-u', '-m', 'deployment.verify_coreml',
        '--checkpoint', str(checkpoint),
        '--package', str(experiment_dir / scheme / 'SongFormerFunctionHead.mlpackage'),
        '--candidate-name', scheme, '--diagnostic', '--limit', '0',
        '--output', str(experiment_dir / f'{scheme}-validation.json')],
        experiment_dir / f'{scheme}-validation.log')


In [ ]:
summary = {'fp32_validation_report': str(proof_path), 'candidates': []}
print(f"FP32 F1: {base['coreml']['macro_f1_present']:.6f}")
for scheme in ('int8', 'palette6'):
    compression = json.loads((experiment_dir / scheme / 'compression_report.json').read_text())
    r = json.loads((experiment_dir / f'{scheme}-validation.json').read_text())
    for key in ('plan_id', 'provenance_id', 'annotations_id', 'checkpoint_sha256', 'tracks'):
        assert r[key] == base[key], f'Perbandingan tidak setara: {key}'
    assert r['full_validation']
    item = dict(scheme=scheme, size_mib=compression['candidate_bytes'] / 1024**2,
        size_reduction_fraction=compression['size_reduction_fraction'],
        macro_f1=r['coreml']['macro_f1_present'],
        f1_delta_from_fp32=r['coreml']['macro_f1_present'] - base['coreml']['macro_f1_present'],
        section_accuracy=r['coreml']['section_accuracy'],
        section_agreement_with_pytorch=r['section_prediction_agreement'],
        numerical_status=r['status'], ready_for_app_integration=False)
    summary['candidates'].append(item)
    print(json.dumps(item, indent=2))
    print('F1 per kelas:', r['coreml']['per_class'])
(experiment_dir / 'comparison.json').write_text(json.dumps(summary, indent=2))
print('Ringkasan:', experiment_dir / 'comparison.json')
print('Belum mengukur speedup atau memori iPhone. Head saja; encoder belum dikompresi.')
